In [5]:
# %pip install openpyxl

In [26]:
import pandas as pd

# Read all sheets from the Excel file
file_path = 'Sample-Superstore.xlsx'

# Read all sheets into a dictionary of DataFrames
excel_data = pd.read_excel(file_path, sheet_name=None, index_col=None)

# Display sheet names
print(f"Found {len(excel_data)} sheets:")
for sheet_name in excel_data.keys():
    print(f"  - {sheet_name}")


Found 3 sheets:
  - Orders
  - People
  - Returns


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [27]:
# Inspect each sheet's structure and data quality
for sheet_name, df in excel_data.items():
    print(f"\n{'='*60}")
    print(f"Sheet: {sheet_name}")
    print(f"{'='*60}")
    print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"\nColumn names and types:")
    print(df.dtypes)
    print(f"\nMissing values:")
    print(df.isnull().sum())
    print(f"\nFirst few rows:")
    display(df.head())


Sheet: Orders

Shape: 9994 rows × 21 columns

Column names and types:
Row ID                    int64
Order ID                 object
Order Date       datetime64[ns]
Ship Date        datetime64[ns]
Ship Mode                object
Customer ID              object
Customer Name            object
Segment                  object
Country                  object
City                     object
State                    object
Postal Code             float64
Region                   object
Product ID               object
Category                 object
Sub-Category             object
Product Name             object
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
dtype: object

Missing values:
Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164



Sheet: People

Shape: 4 rows × 2 columns

Column names and types:
Person    object
Region    object
dtype: object

Missing values:
Person    0
Region    0
dtype: int64

First few rows:


,Person,Region
0,Anna Andreadi,West
1,Chuck Magee,East
2,Kelly Williams,Central
3,Cassandra Brandow,South



Sheet: Returns

Shape: 800 rows × 2 columns

Column names and types:
Returned    object
Order ID    object
dtype: object

Missing values:
Returned    0
Order ID    0
dtype: int64

First few rows:


,Returned,Order ID
0,Yes,CA-2015-100762
1,Yes,CA-2015-100762
2,Yes,CA-2015-100762
3,Yes,CA-2015-100762
4,Yes,CA-2015-100867


In [28]:
# Clean each sheet
cleaned_data = {}

for sheet_name, df in excel_data.items():
    print(f"\nCleaning sheet: {sheet_name}")

    # Make a copy to avoid modifying original
    df_clean = df.copy()

    # 1. Drop row_id column if it exists
    if 'Row ID' in df_clean.columns:
        df_clean = df_clean.drop(columns=['Row ID'])
        print(f"  Dropped 'Row ID' column")

    # 2. Remove completely empty rows and columns
    df_clean = df_clean.dropna(how='all') # drop rows with all NaN values
    df_clean = df_clean.dropna(axis=1, how='all') # drop columns with all NaN values

    # 3. Remove duplicate rows
    duplicates_before = df_clean.duplicated().sum()
    if duplicates_before > 0:
        duplicates = df_clean[df_clean.duplicated()]
        display(duplicates)
    df_clean = df_clean.drop_duplicates(keep='first')
    print(f"  Removed {duplicates_before} duplicate rows")

    # 4. Strip whitespace from string columns
    string_columns = df_clean.select_dtypes(include=['object']).columns
    for col in string_columns:
        df_clean[col] = df_clean[col].str.strip() if df_clean[col].dtype == 'object' else df_clean[col]

    # 5. Handle missing values (you can customize this based on your needs)
    # For numeric columns: you might want to fill with mean/median
    # For categorical columns: you might want to fill with mode or 'Unknown'
    missing_count = df_clean.isnull().sum().sum()
    print(f"  {missing_count} missing values remaining (review and handle as needed)")

    # Store cleaned dataframe
    cleaned_data[sheet_name] = df_clean
    print(f"  Final shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")

print("\n" + "="*60)
print("Data cleaning complete!")
print("="*60)


Cleaning sheet: Orders
  Dropped 'Row ID' column


,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
3406,US-2015-150119,2015-04-23,2015-04-27,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,Ohio,43229.0,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588


  Removed 1 duplicate rows
  11 missing values remaining (review and handle as needed)
  Final shape: 9993 rows × 20 columns

Cleaning sheet: People
  Removed 0 duplicate rows
  0 missing values remaining (review and handle as needed)
  Final shape: 4 rows × 2 columns

Cleaning sheet: Returns


,Returned,Order ID
1,Yes,CA-2015-100762
2,Yes,CA-2015-100762
3,Yes,CA-2015-100762
6,Yes,CA-2015-102652
7,Yes,CA-2015-102652
...,...,...
792,Yes,US-2018-147886
793,Yes,US-2018-147886
794,Yes,US-2018-147886
795,Yes,US-2018-147886


  Removed 504 duplicate rows
  0 missing values remaining (review and handle as needed)
  Final shape: 296 rows × 2 columns

Data cleaning complete!


In [29]:
def to_snake_case(text):
    """Convert text to snake_case"""
    # Replace spaces and hyphens with underscores
    text = text.replace(' ', '_').replace('-', '_')
    # Insert underscore before uppercase letters and convert to lowercase
    text = re.sub('([a-z0-9])([A-Z])', r'\1_\2', text)
    # Remove any duplicate underscores
    text = re.sub('_+', '_', text)
    # Convert to lowercase
    text = text.lower()
    # Remove leading/trailing underscores
    text = text.strip('_')
    return text

# Apply snake_case to all dataframes
print("\n" + "="*60)
print("CONVERTING COLUMN NAMES TO SNAKE_CASE")
print("="*60)

for sheet_name, df in cleaned_data.items():
    print(f"\n{sheet_name}:")
    old_columns = df.columns.tolist()
    new_columns = [to_snake_case(col) for col in old_columns]

    # Show mapping
    for old, new in zip(old_columns, new_columns):
        if old != new:
            print(f"  {old:30s} → {new}")

    # Rename columns
    df.columns = new_columns
    cleaned_data[sheet_name] = df

print("\n✓ All column names converted to snake_case!")


CONVERTING COLUMN NAMES TO SNAKE_CASE

Orders:
  Order ID                       → order_id
  Order Date                     → order_date
  Ship Date                      → ship_date
  Ship Mode                      → ship_mode
  Customer ID                    → customer_id
  Customer Name                  → customer_name
  Segment                        → segment
  Country                        → country
  City                           → city
  State                          → state
  Postal Code                    → postal_code
  Region                         → region
  Product ID                     → product_id
  Category                       → category
  Sub-Category                   → sub_category
  Product Name                   → product_name
  Sales                          → sales
  Quantity                       → quantity
  Discount                       → discount
  Profit                         → profit

People:
  Person                         → person
  Region    

In [30]:
display(cleaned_data.get('Orders'))

,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,CA-2015-110422,2015-01-21,2015-01-23,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,Florida,33180.0,South,FUR-FU-10001889,Furniture,Furnishings,Ultra Door Pull Handle,25.2480,3,0.20,4.1028
9990,CA-2018-121258,2018-02-26,2018-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,California,92627.0,West,FUR-FU-10000747,Furniture,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.9600,2,0.00,15.6332
9991,CA-2018-121258,2018-02-26,2018-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,California,92627.0,West,TEC-PH-10003645,Technology,Phones,Aastra 57i VoIP phone,258.5760,2,0.20,19.3932
9992,CA-2018-121258,2018-02-26,2018-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,California,92627.0,West,OFF-PA-10004041,Office Supplies,Paper,"It's Hot Message Books with Stickers, 2 3/4"" x 5""",29.6000,4,0.00,13.3200


In [34]:
display(cleaned_data.get('Returns').sort_values(by='order_id'))


,returned,order_id
0,Yes,CA-2015-100762
4,Yes,CA-2015-100867
5,Yes,CA-2015-102652
9,Yes,CA-2015-103373
10,Yes,CA-2015-103744
...,...,...
787,Yes,US-2018-136679
789,Yes,US-2018-147886
796,Yes,US-2018-147998
797,Yes,US-2018-151127


In [31]:
# Display summary of cleaned data
print("\n📊 CLEANED DATA SUMMARY\n")

for sheet_name, df in cleaned_data.items():
    print(f"\n{sheet_name}:")
    print(f"  Rows: {df.shape[0]:,}")
    print(f"  Columns: {df.shape[1]}")
    print(f"  Column names: {', '.join(df.columns.tolist())}")

print("\n✅ Your data is now ready for analytics!")
print("\nYou can access the cleaned data using:")
for sheet_name in cleaned_data.keys():
    print(f"  cleaned_data['{sheet_name}']")


📊 CLEANED DATA SUMMARY


Orders:
  Rows: 9,993
  Columns: 20
  Column names: order_id, order_date, ship_date, ship_mode, customer_id, customer_name, segment, country, city, state, postal_code, region, product_id, category, sub_category, product_name, sales, quantity, discount, profit

People:
  Rows: 4
  Columns: 2
  Column names: person, region

Returns:
  Rows: 296
  Columns: 2
  Column names: returned, order_id

✅ Your data is now ready for analytics!

You can access the cleaned data using:
  cleaned_data['Orders']
  cleaned_data['People']
  cleaned_data['Returns']
